# SQL

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/database/drug-food-interaction.db")

c = conn.cursor()

# Load cleaned dataset
df = pd.read_csv("../data/clean_data/clean_ddinter_data.csv")



## Tạo table(3NF)


In [2]:
c.execute("""
CREATE TABLE IF NOT EXISTS Drugs (
    drug_id INTEGER PRIMARY KEY AUTOINCREMENT,
    drug_name TEXT UNIQUE
)
""")

c.execute("""
CREATE TABLE IF NOT EXISTS Foods (
    food_id INTEGER PRIMARY KEY AUTOINCREMENT,
    food_name TEXT UNIQUE
)
""")

c.execute("""
CREATE TABLE IF NOT EXISTS Interactions (
    interaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    drug_id INTEGER,
    food_id INTEGER,
    severity TEXT,
    description TEXT,
    source TEXT,
    FOREIGN KEY (drug_id) REFERENCES Drugs(drug_id),
    FOREIGN KEY (food_id) REFERENCES Foods(food_id)
)
""")

conn.commit()

## Chèn dữ liệu vào table

In [3]:
# tạo biến lấy dữ liệu của cột tên drug và food loại bỏ trùng lặp 
# Mục đích để khi insert dữ liệu vào Bảng Drugs và Food các row sẽ unique

drugs = df['drug_name'].drop_duplicates().reset_index(drop = True)
foods = df['food_name'].drop_duplicates().reset_index(drop = True)


In [4]:
#Nạp dữ liệu vào table Drugs
for drug in drugs:
    c.execute("""
        INSERT OR IGNORE INTO Drugs(drug_name)
        VALUES (?)
""", (drug,))
    
#Nạp dữ liệu vào table Foods
for food in foods:
    c.execute("""
        INSERT OR IGNORE INTO Foods (food_name)
        VALUES (?)
    """, (food,))

# CREATE ID MAPPING (Tạo từ điển ID)
# VD: {"Paracetamol": 1, "Aspirin": 2, "Ibuprofen": 3}
c.execute("""SELECT drug_id,drug_name FROM Drugs""")
drug_map = {name:id for id,name in c.fetchall()} 

c.execute("SELECT food_id, food_name FROM Foods")
food_map = {name: id for id, name in c.fetchall()}

#INSERT INTERACTIONS
for _,row in df.iterrows(): # duyệt từng dòng trong df( _ là bỏ qua index)
    drug_id = drug_map[row['drug_name']] 
    food_id = food_map[row['food_name']]

    c.execute("""
        INSERT OR IGNORE INTO Interactions(drug_id, food_id, severity, description, source)
        VALUES (?, ?, ?, ?, ?)
    """, (drug_id, food_id, row['severity'], row['description'], row['source']))
    

conn.commit()

In [5]:
# Test truy vấn 
query = """
SELECT 
    d.drug_name,
    f.food_name,
    i.severity,
    i.description
FROM Interactions i
JOIN Drugs d ON i.drug_id = d.drug_id
JOIN Foods f ON i.food_id = f.food_id
LIMIT 10
"""

result = pd.read_sql_query(query, conn)
print(result)

                            drug_name         food_name  severity  \
0                     calcium lactate           spinach  Moderate   
1                     calcium lactate           rhubarb  Moderate   
2                     calcium lactate              bran  Moderate   
3                     calcium lactate             grain  Moderate   
4                         pralsetinib     highfat foods      High   
5                         pralsetinib  grapefruit juice      High   
6  tetraferric tricitrate decahydrate     highfat foods  Moderate   
7                         pirfenidone  grapefruit juice  Moderate   
8            apraclonidine ophthalmic           alcohol  Moderate   
9                        olutasidenib     highfat foods  Moderate   

                                         description  
0  Administration with food may increase the abso...  
1  Administration with food may increase the abso...  
2  Administration with food may increase the abso...  
3  Administration wi

## Các lệnh truy vấn

In [ ]:
#1. Thống kê severity
pd.read_sql("""
SELECT severity, COUNT(*) as count
FROM Interactions
GROUP BY severity
""", conn)



,severity,count
0,High,105
1,Low,38
2,Moderate,714


In [ ]:
#2. Top 10 thực phẩm gây tương tác nhiều nhất

pd.read_sql("""
    SELECT f.food_name, COUNT(*) AS Total
    FROM Interactions i
    JOIN Foods f ON f.food_id = i.food_id
    GROUP BY f.food_name
    ORDER BY Total DESC 
    LIMIT 10

""", conn)


,food_name,Total
0,alcohol,448
1,grapefruit juice,154
2,highfat foods,76
3,food,66
4,food high in potassium,18
5,dairy products,16
6,tyrosinerich foods,12
7,coffee,10
8,orange juice,8
9,cigarette,7


In [ ]:
#3. Top thuốc rủi ro cao
pd.read_sql("""
SELECT d.drug_name, COUNT(*) as count
FROM Interactions i
JOIN Drugs d ON i.drug_id = d.drug_id
WHERE i.severity = 'High'
GROUP BY d.drug_name
ORDER BY count DESC
LIMIT 10
""", conn)

,drug_name,count
0,pexidartinib,3
1,pralsetinib,2
2,pimozide,2
3,lovastatin,2
4,lemborexant,2
5,aluminum hydroxide,2
6,zanubrutinib,1
7,voclosporin,1
8,venetoclax,1
9,valbenazine,1


In [ ]:
conn.close()